In [ ]:
import itertools

import pandas as pd
import networkx as nx
import plotly.io as pio
import plotly.graph_objects as go

In [ ]:
pio.renderers.default = "browser"

In [ ]:
path_json = "parameterized_dataset_udi.json"
df_graph = pd.read_json(path_json)
df_graph.head()

In [ ]:
G = nx.Graph()

for _, linha in df_graph.iterrows():
    especie = linha['nome_pt']

    # Taxonomia
    G.add_edge(especie, linha['familia'], tipo='pertence a')
    G.add_edge(linha['genero'], linha['subfamilia'], tipo='pertence a')
    G.add_edge(linha['subfamilia'], linha['familia'], tipo='pertence a')
    G.add_edge(linha['familia'], linha['ordem'], tipo='pertence a')

    # Morfologia e ecologia
    # G.add_edge(especie, linha['tipo_bico'], tipo='tipo de bico')
    # G.add_edge(especie, linha['caracteristica_trofica'], tipo='trófico')

    # Dieta (lista)
    # if isinstance(linha['dieta_principal'], list):
    #    for dieta in linha['dieta_principal']:
    #         G.add_edge(especie, dieta, tipo='dieta')
    # else:
    #     if pd.notna(linha['dieta_principal']):
    #         G.add_edge(especie, linha['dieta_principal'], tipo='dieta')

    # Habitat (lista)
    # if isinstance(linha['habitat'], list):
    #     for hab in linha['habitat']:
    #         G.add_edge(especie, hab, tipo='habitat')
    # else:
    #     if pd.notna(linha['habitat']):
    #         G.add_edge(especie, linha['habitat'], tipo='habitat')
    
    # Cores (lista)
    # if isinstance(linha['cores'], list):
    #     for cor in linha['cores']:
    #         G.add_edge(especie, cor, tipo='cor')
    # else:
    #     if pd.notna(linha['cores']):
    #         G.add_edge(especie, linha['cores'], tipo='cor')

In [ ]:
pos = nx.spring_layout(G, seed=42)

# Traço das arestas
edges_x, edges_y = [], []
for edge in G.edges():
    x0, y0 = pos[edge[0]]
    x1, y1 = pos[edge[1]]
    edges_x.extend([x0, x1, None])
    edges_y.extend([y0, y1, None])

edge_trace = go.Scatter(
    x=edges_x, y=edges_y,
    line=dict(width=1, color='gray'),
    hoverinfo='none',
    mode='lines'
)

# Nós
nodes_x, nodes_y, node_text = [], [], []
for node in G.nodes():
    x, y = pos[node]
    nodes_x.append(x)
    nodes_y.append(y)
    node_text.append(str(node))

node_trace = go.Scatter(
    x=nodes_x, y=nodes_y,
    mode='markers+text',
    text=[txt for txt in node_text],
    textposition='top center',
    hovertext=node_text,
    marker=dict(color='lightblue', size=10, line_width=1)
)

fig = go.Figure(data=[edge_trace, node_trace],
    layout=go.Layout(
        title='Grafo das Espécies de Aves e suas Características',
        # titlefont_size=16,
        showlegend=False,
        hovermode='closest',
        width=1000,
        height=800,
        margin=dict(b=20, l=5, r=5, t=40),
        xaxis=dict(showgrid=False, zeroline=False),
        yaxis=dict(showgrid=False, zeroline=False)
    )
)

fig.show()